# Building the model

This notebook just wires up the model and pushes one batch of real data through it,
to confirm the shapes all line up before spending time on a real training run. It
doesn't train anything.

## The architecture

Each vertex's input is 6 numbers: its 3D position (normalized into a unit box) and
its surface normal (a 3D unit vector, computed from the mesh since the dataset
doesn't provide one). Three `FeaStConv` graph convolution layers turn this into a
256-dimensional embedding per vertex, which then splits into two heads:

- a **pressure head** — a small per-vertex MLP, giving one pressure value per vertex
- a **Cd head** — mean+max pooling across the whole car's vertices, then an MLP,
  giving one drag value for the whole shape

See `docs/diagrams/model_architecture.svg` (and the README) for the full diagram.

## Why FeaStConv

A normal image convolution works because pixels have a fixed neighborhood layout —
"the pixel above me" always means the same direction, so a kernel can learn one
weight per neighbor position. A mesh vertex has no such fixed layout: one vertex's
neighbor might sit to its left, another vertex's neighbor to its front, with no
shared frame of reference.

`FeaStConv` handles this by learning several weight matrices per layer (4 here,
called "heads"), plus a small function that looks at the *relative position* between
two connected vertices and decides how much of that edge's message should go through
each weight matrix. In effect, the network learns its own notion of "neighbor
direction" instead of relying on a fixed one — a reasonable stand-in for the
orientation-aware, geometry-respecting convolutions used in the paper this project is
based on.


Load 4 cached cars and batch them into one `DataBatch` — PyG's way of packing several
graphs of different sizes into a single tensor for the GPU. `batch.batch` is what
lets the model later know which vertices belong to which car, e.g. when pooling for
the Cd head.


In [1]:
import sys
sys.path.insert(0, "..")

import torch
from torch_geometric.loader import DataLoader
from src.model import MeshSurrogate

train_data = torch.load("../outputs/cache/train_pyg.pt", weights_only=False)
loader = DataLoader(train_data[:4], batch_size=4)
batch = next(iter(loader))
print(batch)


MeshDataBatch(x=[14344, 6], edge_index=[2, 86016], face=[3, 28672], face_batch=[28672], y_pressure=[14344], y_cd=[4], sample_id=[4], batch=[14344], ptr=[5])


Untrained forward pass. The shapes are what matter here: `pressure_pred` should have
one value per vertex across the whole batch (14344 = 4 cars × ~3586 vertices each),
and `cd_pred` should have exactly one value per car (4).


In [2]:
model = MeshSurrogate(in_channels=6)
pressure_pred, cd_pred = model(batch.x, batch.edge_index, batch.batch)
print("pressure_pred", pressure_pred.shape)
print("cd_pred", cd_pred.shape)
assert pressure_pred.shape[0] == batch.x.shape[0]
assert cd_pred.shape[0] == batch.num_graphs


pressure_pred torch.Size([14344])
cd_pred torch.Size([4])
